# Fine-tuning Pipeline (MACE-MH-1 OMAT)

This notebook uses multi-head tuning with 2 datasets:
    * DFT relaxation set constructed below
    * configs from OMAT head w/ energy + force labels from MACE-MH-1 predictions (to preserve original model behavior)
      replay source [06/23/26]: https://github.com/ACEsuit/mace-foundations/releases/tag/mace_mh_1

See https://mace-docs.readthedocs.io/en/latest/guide/multihead_finetuning.html for more details.

## Conda Env Setup
In the CLI, run 
`path/mace-mh-1.yml`

In [20]:
from ase.io import iread, read, write
from pathlib import Path

# USER INPUT HERE: set cache folder for the dataset being selected from
CACHE = "cache/dropbox_O2DissociationSAA_2"
# CACHE = "cache/box_EO_Project_reactivity"

PREFIX = "withNi_"  # cache file prefix

In [11]:
# test different energy read-in methods for consistency

frame = read("/Users/zschwab/vscode/montemore-research/finetuned-mlips/data-processing/outcars/Acetaldehyde_Ni0O0Ag_OUTCAR", index=-1, format="vasp-out")
print("calc.results['energy']:", frame.calc.results["energy"])
print("get_potential_energy() default:", frame.get_potential_energy())
print("get_potential_energy(force_consistent=True):", frame.get_potential_energy(force_consistent=True))

calc.results['energy']: -142.47214035
get_potential_energy() default: -142.47214035
get_potential_energy(force_consistent=True): -142.47487253


In [21]:
# read in list of selected DFT run folder names from cache
foldernames = []

with open(f"../data-processing/{CACHE}/{PREFIX}screened_foldernames.txt", "r") as f:
    foldernames = [line.strip() for line in f]

print(f"Total items: {len(foldernames)}")

Total items: 263


In [22]:
from collections import defaultdict
import re

def make_outcar_paths(foldernames):
    outcar_dir = Path("../data-processing/outcars/")
    
    groups = defaultdict(list)

    for p in outcar_dir.iterdir():
        m = re.match(r"(.+)_OUTCAR(?:-(\d+))?$", p.name)
        if m:
            name, num = m.group(1), m.group(2)
            if name not in foldernames:
                continue
            groups[name].append((int(num) if num else float("inf"), p))

    outcar_paths = []
    for name, files in groups.items():
        files.sort(key=lambda x: x[0])
        outcar_paths.append((name, [f[1] for f in files]))

    return outcar_paths

# OUTCARP_PATHS: list of paths to OUTCAR file for each run
OUTCAR_PATHS = make_outcar_paths(foldernames)

In [23]:
print(f"Matched {len(OUTCAR_PATHS)} of {len(foldernames)} screened folders")
missing = set(foldernames) - {name for name, _ in OUTCAR_PATHS}
if missing:
    print(f"Missing from outcar_dir: {missing}")

Matched 263 of 263 screened folders


In [27]:
# patch malformed OUTCARs so ASE parser works
import tempfile

def _sanitize_outcar(path):
    # strip malformed 'NBANDS=' lines (blank value, seen in restart/continuation OUTCAR segments) that crash ASE's vasp-out parser
    with open(path, "r", errors="ignore") as f:
        lines = f.readlines()
    # drop any NBANDS= line that has no digit following it
    cleaned = [ln for ln in lines if not ("NBANDS=" in ln and not re.search(r"NBANDS=\s*\d", ln))]
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix="_OUTCAR", delete=False)
    tmp.writelines(cleaned)
    tmp.close()
    return tmp.name

In [28]:
# create finetune set by storing the first and last frames of each DFT run as an ASE Atoms object

def load_frames(paths):
    # return tuple of first and last frames of DFT run
    _, outcar_paths = paths
    
    # strip malformed 'NBANDS=' lines
    clean_paths = [_sanitize_outcar(p) for p in outcar_paths]
    
    # grab 4th frame as "first" to let calculations evolve from initial guess
    i = 0
    found = False
    for p in clean_paths:  # continue to next OUTCAR if first has <4 frames
        for frame in iread(p, format="vasp-out"):
            first = frame
            if i == 3:
                found = True
                break
            i += 1
        if found:
            break
    if not found:
        print(f"<4 frames total across {paths} (only {i+1})")
    last = read(outcar_paths[-1], index=-1, format="vasp-out")

    return first, last

In [25]:
# test OUTCAR read-in here
print(OUTCAR_PATHS[0])
test_frame = load_frames(OUTCAR_PATHS[0])
print(test_frame)
print(test_frame[0].get_total_energy())
print(test_frame[1].get_total_energy())

('CO3_Osub_Ag100_alt2', [PosixPath('../data-processing/outcars/CO3_Osub_Ag100_alt2_OUTCAR-1'), PosixPath('../data-processing/outcars/CO3_Osub_Ag100_alt2_OUTCAR')])
(Atoms(symbols='Ag36CO4', pbc=True, cell=[8.62093981, 8.62093981, 26.095925], calculator=SinglePointDFTCalculator(...)), Atoms(symbols='Ag36CO4', pbc=True, cell=[8.62093981, 8.62093981, 26.095925], calculator=SinglePointDFTCalculator(...)))
-134.73198543
-135.42129045


In [ ]:
dft_frames_set = []
HEAD_NAME = "dft_ag"

for path in OUTCAR_PATHS:
    try:
        first, last = load_frames(path)
        for atoms in [first, last]:
            # atoms = atoms.get_potential_energy()
            atoms.info["REF_energy"] = atoms.calc.results["energy"]
            atoms.info["head"] = HEAD_NAME
            atoms.arrays["REF_forces"] = atoms.calc.results["forces"]
        dft_frames_set.extend([first, last])
    except Exception as e:
        print(f"FAILED: {path} -> {e}")
        continue

dataset = "O2DissociationSAA_2"  
# dataset = "EO_Project_reactivity"

write(f"../training-data/{PREFIX}_{dataset}_input.xyz", dft_frames_set, format="extxyz")

In [ ]:
# remove malformed runs from unhandled cases

# from O2DissociationSAA_2 dataset:
bad_runs = ["3OMC_Ni1Ag_alt0", "3OMC_Ni1Ag_alt2", "OCCO_2OMC_Ni1Ag2", "4OMC_NiAg_0", "Ag_rec_0_OCCO_more_0", "OCCO_2OMC_Ni1Ag3", "OCCO_2OMC_Ni1Ag0", "3OMC_Ni1Ag_alt1", "4OMC_NiAg_1", "OCCO_2OMC_Ni1Ag1"]
print(f"malformed runs: {len(bad_runs)} / {len(OUTCAR_PATHS)}")


malformed runs: 10 / 263


In [32]:
OUTCAR_PATHS = [
    (folder, paths) for folder, paths in OUTCAR_PATHS
    if folder not in bad_runs
]

Check the .xyz file of training data you just wrote.

In [18]:
# check frame counts

check = read(f"../training-data/{PREFIX}_{dataset}_input.xyz", index=":")
print(f"{len(OUTCAR_PATHS)} paths to DFT runs --> 2 * {len(OUTCAR_PATHS)} - 2 * FAILED = ")
print(len(check), "frames written")
print(check[0].info)          # should show REF_energy, head
print(check[0].arrays.keys()) # should include REF_forces

178 paths to DFT runs --> 2 * 178 - 2 * FAILED = 
356 frames written
{'REF_energy': np.float64(-347.7501213), 'head': 'dft_ag'}
dict_keys(['numbers', 'positions', 'REF_forces'])


In [19]:
# check head label consistency

set(a.info["head"] for a in check) == {"dft_ag"}

True

**Concatenate .xyz files** from multiple datasets into one with the following CLI command (run in finetuned-mlips/training-data/)

`cat EO_Project_reactivity-input.xyz <(echo) O2DissociationSAA_2-input.xyz > train.xyz`

**Create the combined finetuning set** with the following CLI command.
Run in the finetuned-mlips/training-data/ folder.
This calls fine_tuning_select.py (stored in the mace-torch package you installed in your conda env) 
to create a filtered 3rd file from your dft data and the replay. Edit flags below filepaths at will.

See all possible parameters and more info w/ `python -m mace.cli.fine_tuning_select --help`

```bash 
python -m mace.cli.fine_tuning_select \
  --configs_pt data/replay-data-mh-1-omat-pbe.xyz \
  --configs_ft data/train.xyz \
  --num_samples 10000 \
  --subselect fps \
  --model path/to/foundation_model.model \
  --output selected_configs.xyz \
  --filtering_type combinations \
  --head_pt omat_pbe \
  --head_ft tuned_head \
  --weight_pt 1.0 \
  --weight_ft 10.0
```
Parameters:
* `--configs_pt`: Path to the replay dataset
* `--configs_ft`: Path to your target dataset
* `--num_samples`: Number of configurations to select from the replay dataset
* `--subselect`: Method for subselection (fps for Farthest Point Sampling or random)
* `--filtering_type`: How to filter configurations based on elements:
    * `combinations`: Keep configurations with combinations of elements in your target dataset
    * `exclusive`: Keep configurations containing only elements in your target dataset
    * `inclusive`: Keep configurations containing all elements in your target dataset
    * `none`: No filtering
* `--filter_atomic_numbers_pt`: Optionally specify specific atomic numbers to filter by
* `--weight_pt`: weights the loss calculated on replay (pretraining refs) structures
* `--weight_ft`: weights the loss calcualted on finetune structures

Copy/Paste Version:
```bash
python -m mace.cli.fine_tuning_select --configs_pt ../training-data/replay-data-mh-1-omat-pbe.xyz --configs_ft ../training-data/train.xyz --num_samples 811 --subselect fps --model /Users/zschwab/.cache/huggingface/hub/models--mace-foundations--mace-mh-1/snapshots/95fc198fe533fd351b4b74bd0fdbdc6c10a2dddd/mace-mh-1.model --output ../training-data/selected_configs.xyz --filtering_type combinations --filter_atomic_numbers_pt "[1, 6, 8, 47]" --head_pt omat_pbe --head_ft dft_ag --weight_pt 1.0 --weight_ft 10.0 --default_dtype float32 --disallow_random_padding
```

**Verify replay mode**: Mode 1 (DFT) vs Mode 2 (model-labeled)

Checks whether `selected_configs.xyz` (subsampled from replay dataset) contains true DFT labels (Mode 1) or foundation model-generated labels (Mode 2) by comparing a replay frame's energy/forces to the omat_pbe head zero-shot. If they match within fp64 precision, the labels were generated by the model, confirming Mode 2.

In [ ]:
from mace.calculators import MACECalculator
import numpy as np

# load a frame from the replay/pt file
frame = read("../training-data/selected_configs.xyz", index=0)

MODEL_PATH = "/Users/zschwab/.cache/huggingface/hub/models--mace-foundations--mace-mh-1/snapshots/95fc198fe533fd351b4b74bd0fdbdc6c10a2dddd/mace-mh-1.model"

# foundation model, raw omat_pbe head (no finetuning)
calc = MACECalculator(
    model_paths=MODEL_PATH,      # or full path to the mace-mh-1 .model file
    # device="cuda",
    head="omat_pbe"
)
frame.calc = calc

pred_E = frame.get_potential_energy()
pred_F = frame.get_forces()

label_E = frame.info["REF_energy"]
label_F = frame.arrays["REF_forces"]

print(f"Predicted E: {pred_E:.6f} eV  |  Label E: {label_E:.6f} eV  |  ΔE: {abs(pred_E-label_E):.6e}")
print(f"Max |ΔF|: {np.max(np.abs(pred_F - label_F)):.6e} eV/Å")

# (result confirms that this is mode 2, b/c err is a floating point rounding value.)

/Users/zschwab/miniconda3/envs/mace-mh-1/lib/python3.11/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Predicted E: -7.233528 eV  |  Label E: -7.233528 eV  |  ΔE: 8.881784e-16
Max |ΔF|: 4.595095e-09 eV/Å


**Zero-shot error** of omat_pbe on train.xyz (distribution-shift check)

In [ ]:
frames = read("../training-data/train.xyz", index=":")

calc = MACECalculator(
    model_paths=MODEL_PATH,
    # device="cuda",
    head="omat_pbe"
)

dE_per_atom = []
dF_max = []

for atoms in frames:
    atoms_copy = atoms.copy()
    atoms_copy.calc = calc

    pred_E = atoms_copy.get_potential_energy()
    pred_F = atoms_copy.get_forces()

    label_E = atoms.info["REF_energy"]
    label_F = atoms.arrays["REF_forces"]

    n_atoms = len(atoms)
    dE_per_atom.append(abs(pred_E - label_E) / n_atoms)
    dF_max.append(np.max(np.abs(pred_F - label_F)))

dE_per_atom = np.array(dE_per_atom) * 1000  # meV/atom
dF_max = np.array(dF_max) * 1000            # meV/A

print(f"N frames: {len(frames)}")
print(f"RMSE E/atom: {np.sqrt(np.mean(dE_per_atom**2)):.1f} meV  (mean: {dE_per_atom.mean():.1f}, max: {dE_per_atom.max():.1f})")
print(f"Mean max|dF|: {dF_max.mean():.1f} meV/A  (max: {dF_max.max():.1f})")

# flag worst outliers
worst = np.argsort(dE_per_atom)[-5:][::-1]
print("\nWorst 5 frames by E error:")
for i in worst:
    print(f"  idx {i}: dE={dE_per_atom[i]:.1f} meV/atom, formula={frames[i].get_chemical_formula()}")

/Users/zschwab/miniconda3/envs/mace-mh-1/lib/python3.11/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


N frames: 318
RMSE E/atom: 273.3 meV  (mean: 272.4, max: 313.5)
Mean max|dF|: 248.6 meV/A  (max: 734.1)

Worst 5 frames by E error:
  idx 243: dE=313.5 meV/atom, formula=Ag74O8
  idx 242: dE=313.5 meV/atom, formula=Ag74O8
  idx 278: dE=305.3 meV/atom, formula=C2H4Ag76O6
  idx 69: dE=305.2 meV/atom, formula=Ag68O8
  idx 68: dE=305.1 meV/atom, formula=Ag68O8


**Check element composition breakdown in replay dataset** 

In [ ]:
from collections import Counter

atoms_list = read("../training-data/selected_configs.xyz", index=":")

counts = Counter(frozenset(a.get_chemical_symbols()) for a in atoms_list)

for combo, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{n:6d}  {'-'.join(sorted(combo))}")

print(f"\ntotal: {len(atoms_list)}")

no_ag_total = sum(n for combo, n in counts.items() if "Ag" not in combo)
print(f"no Ag: {no_ag_total}    contains Ag: {len(atoms_list) - no_ag_total}")

**Get EOs** (energies of single isolated atoms) for reference

In [4]:
# USER INPUT HERE: enter atoms
ATOMS = ["Ag", "C", "H", "O"]

REF_PATH = "../training-data/dft-data/"

def get_eo(atom):
    frame = read(f"{REF_PATH}{atom}_Gas_OUTCAR", index=-1, format="vasp-out")
    energy = frame.get_potential_energy()
    print(f"{atom}: {energy} eV")
    return energy

In [5]:
for a in ATOMS:
    get_eo(a)

Ag: -0.19458509 eV
C: -1.28369203 eV
H: -1.11167391 eV
O: -1.55774442 eV


**Train model on combined set** with the following CLI command/bash script.

```bash
python -m mace.cli.run_train \
--name Ag_CHO_finetuned_mace-mh-1 \
--train_file data/train.xyz \
--foundation_model mh-1 \
--foundation_head omat_pbe \
--pt_train_file data/selected_configs.xyz \
--energy_weight 1.0 \
--forces_weight 100.0 \
--swa \
--swa_energy_weight 10.0 \
--swa_forces_weight 100.0 \
--stress_weight 0.0 \
--lr 5e-4 \
--multiheads_finetuning True \
--valid_fraction 0.1 \
--E0s "{47: -0.19458509, 8: -1.55774442, 6: -1.28369203, 1: -1.11167391}" \
--max_num_epochs 30 \
--device "cuda" \
```

## Training Guide

**run_train flags** \
`--train-file` --> the target/finetuning dataset you created (becomes Default head) \
`--pt-train-file` --> the replay/pretraining dataset, created by selecting from the .xyz of replay configs (becomes pt_head) 

**files** \
`selected_configs.xyz` --> subsampled from replay data according to the criteria set (eg. atomic numbers) \
`train.xyz` --> your finetuning dataset \
`selected_configs_combined` --> `train.xyz` + replay data from `selected_configs.xyz`